<h1>Webscraping</h1>

In this exercise, you'll practice using BeautifulSoup to parse the content of a web page. The page that you'll be scraping, https://realpython.github.io/fake-jobs/, contains job listings. Your job is to extract the data on each job and convert into a pandas DataFrame.

<h2>Start by performing a GET request on the url above and convert the response into a BeautifulSoup object.</h2>

In [45]:
import requests
import io
import pandas as pd
import re
from bs4 import BeautifulSoup as BS

In [2]:
url = "https://realpython.github.io/fake-jobs/"
response = requests.get(url)

In [3]:
soup = BS(response.text)

<h3>a. Use the .find method to find the tag containing the first job title ("Senior Python Developer"). Hint: can you find a tag type and/or a class that could be helpful for extracting this information? Extract the text from this title.</h3>

In [4]:
spd = soup.find('h2')
spd

<h2 class="title is-5">Senior Python Developer</h2>

<h3>b. Now, use what you did for the first title, but extract the job title for all jobs on this page. Store the results in a list.</h3>

In [5]:
job_titles = soup.find_all('h2', attrs={'class' : 'title is-5'})
job_titles[0:6]

[<h2 class="title is-5">Senior Python Developer</h2>,
 <h2 class="title is-5">Energy engineer</h2>,
 <h2 class="title is-5">Legal executive</h2>,
 <h2 class="title is-5">Fitness centre manager</h2>,
 <h2 class="title is-5">Product manager</h2>,
 <h2 class="title is-5">Medical technical officer</h2>]

In [6]:
def extract_text(list):
    newlist = []
    for entry in list:
        clean_entry = entry.text
        newlist.append(clean_entry.strip())
    return newlist

In [7]:
titles_clean = extract_text(job_titles)
titles_clean[0:5]

['Senior Python Developer',
 'Energy engineer',
 'Legal executive',
 'Fitness centre manager',
 'Product manager']

<h3>Finally, extract the companies, locations, and posting dates for each job.</h3>

For example, the first job has a company of "Payne, Roberts and Davis", a location of "Stewartbury, AA", and a posting date of "2021-04-08". Ensure that the text that you extract is clean, meaning no extra spaces or other characters at the beginning or end.

In [8]:
companies = soup.find_all('h3', attrs={'class' : 'subtitle is-6 company'})
companies[0:5]

[<h3 class="subtitle is-6 company">Payne, Roberts and Davis</h3>,
 <h3 class="subtitle is-6 company">Vasquez-Davidson</h3>,
 <h3 class="subtitle is-6 company">Jackson, Chambers and Levy</h3>,
 <h3 class="subtitle is-6 company">Savage-Bradley</h3>,
 <h3 class="subtitle is-6 company">Ramirez Inc</h3>]

In [9]:
companies_clean = extract_text(companies)
companies_clean[0:5]

['Payne, Roberts and Davis',
 'Vasquez-Davidson',
 'Jackson, Chambers and Levy',
 'Savage-Bradley',
 'Ramirez Inc']

In [10]:
locations = soup.find_all('p', attrs={'class' : 'location'})
locations[0]

<p class="location">
        Stewartbury, AA
      </p>

In [11]:
locations_clean = extract_text(locations)

In [17]:
locations_clean[0]

'Stewartbury, AA'

In [18]:
dates = soup.find_all('time')
dates[0:5]

[<time datetime="2021-04-08">2021-04-08</time>,
 <time datetime="2021-04-08">2021-04-08</time>,
 <time datetime="2021-04-08">2021-04-08</time>,
 <time datetime="2021-04-08">2021-04-08</time>,
 <time datetime="2021-04-08">2021-04-08</time>]

In [20]:
dates_clean = extract_text(dates)
dates_clean[0:5]

['2021-04-08', '2021-04-08', '2021-04-08', '2021-04-08', '2021-04-08']

<h3>d. Take the lists that you have created and combine them into a pandas DataFrame.</h3>

In [27]:
job_listings = pd.DataFrame()

In [28]:
job_listings['title'] = titles_clean
job_listings['company'] = companies_clean
job_listings['location'] = locations_clean
job_listings['post_date'] = dates_clean

In [29]:
job_listings.head()

,title,company,location,post_date
0,Senior Python Developer,"Payne, Roberts and Davis","Stewartbury, AA",2021-04-08
1,Energy engineer,Vasquez-Davidson,"Christopherville, AA",2021-04-08
2,Legal executive,"Jackson, Chambers and Levy","Port Ericaburgh, AA",2021-04-08
3,Fitness centre manager,Savage-Bradley,"East Seanview, AP",2021-04-08
4,Product manager,Ramirez Inc,"North Jamieview, AP",2021-04-08


<h2>Next, add a column that contains the url for the "Apply" button. Try this in two ways.</h2>

<h3>a. First, use the BeautifulSoup find_all method to extract the urls.</h3>

In [42]:
apply_links = soup.find_all('a', string = 'Apply')
#apply_links

In [39]:
def get_urls(list):
    urls = []
    for entry in list:
        urls.append(entry.get('href'))
    return urls

In [41]:
links_clean = get_urls(apply_links)
links_clean[0:5]

['https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html',
 'https://realpython.github.io/fake-jobs/jobs/energy-engineer-1.html',
 'https://realpython.github.io/fake-jobs/jobs/legal-executive-2.html',
 'https://realpython.github.io/fake-jobs/jobs/fitness-centre-manager-3.html',
 'https://realpython.github.io/fake-jobs/jobs/product-manager-4.html']

<h3>b. Next, get those same urls in a different way. Examine the urls and see if you can spot the pattern of how they are constructed. Then, build the url using the elements you have already extracted.</h3>

Ensure that the urls that you created match those that you extracted using BeautifulSoup. Warning: You will need to do some string cleaning and prep in constructing the urls this way. For example, look carefully at the urls for the "Software Engineer (Python)" job and the "Scientist, research (maths)" job.

In [60]:
def gen_urls(data):
    urls = []
    for index, row in data.iterrows():
        url_title = re.sub('[,/()]', ' ', row['title']).lower()
        url_title = "-".join(url_title.split())
        urls.append('https://realpython.github.io/fake-jobs/jobs/' + str(url_title) + '-' + str(index) + '.html')
    return urls

In [61]:
job_listings['link'] = gen_urls(job_listings)
job_listings.head()

,title,company,location,post_date,link
0,Senior Python Developer,"Payne, Roberts and Davis","Stewartbury, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/se...
1,Energy engineer,Vasquez-Davidson,"Christopherville, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/en...
2,Legal executive,"Jackson, Chambers and Levy","Port Ericaburgh, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/le...
3,Fitness centre manager,Savage-Bradley,"East Seanview, AP",2021-04-08,https://realpython.github.io/fake-jobs/jobs/fi...
4,Product manager,Ramirez Inc,"North Jamieview, AP",2021-04-08,https://realpython.github.io/fake-jobs/jobs/pr...


<h2>Finally, we want to get the job description text for each job.</h2>

<h3>a. Start by looking at the page for the first job, https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html. Using BeautifulSoup, extract the job description paragraph.</h3>

In [67]:
test_url = "https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html"
test_response = requests.get(test_url)
test_soup = BS(test_response.text)

In [76]:
test_desc = test_soup.find_all('p')
clean_desc = test_desc[1].text
clean_desc

'Professional asset web application environmentally friendly detail-oriented asset. Coordinate educational dashboard agile employ growth opportunity. Company programs CSS explore role. Html educational grit web application. Oversea SCRUM talented support. Web Application fast-growing communities inclusive programs job CSS. Css discussions growth opportunity explore open-minded oversee. Css Python environmentally friendly collaborate inclusive role. Django no experience oversee dashboard environmentally friendly willing to learn programs. Programs open-minded programs asset.'

<h3>b. We want to be able to do this for all pages. Write a function which takes as input a url and returns the description text on that page.</h3>

For example, if you input "https://realpython.github.io/fake-jobs/jobs/television-floor-manager-8.html" into your function, it should return the string "At be than always different American address. Former claim chance prevent why measure too. Almost before some military outside baby interview. Face top individual win suddenly. Parent do ten after those scientist. Medical effort assume teacher wall. Significant his himself clearly very. Expert stop area along individual. Three own bank recognize special good along.".

In [79]:
def get_desc(url):
    url_response = requests.get(url)
    url_soup = BS(url_response.text)
    result = url_soup.find_all('p')[1].text
    return result

print(get_desc('https://realpython.github.io/fake-jobs/jobs/television-floor-manager-8.html'))

At be than always different American address. Former claim chance prevent why measure too. Almost before some military outside baby interview. Face top individual win suddenly. Parent do ten after those scientist. Medical effort assume teacher wall. Significant his himself clearly very. Expert stop area along individual. Three own bank recognize special good along.


<h3>c. Use the [.apply method](https://pandas.pydata.org/docs/reference/api/pandas.Series.apply.html) on the url column you created above to retrieve the description text for all of the jobs.</h3>

In [80]:
job_listings['description'] = job_listings['link'].apply(get_desc)
job_listings.head()

,title,company,location,post_date,link,description
0,Senior Python Developer,"Payne, Roberts and Davis","Stewartbury, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/se...,Professional asset web application environment...
1,Energy engineer,Vasquez-Davidson,"Christopherville, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/en...,Party prevent live. Quickly candidate change a...
2,Legal executive,"Jackson, Chambers and Levy","Port Ericaburgh, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/le...,Administration even relate head color. Staff b...
3,Fitness centre manager,Savage-Bradley,"East Seanview, AP",2021-04-08,https://realpython.github.io/fake-jobs/jobs/fi...,Tv program actually race tonight themselves tr...
4,Product manager,Ramirez Inc,"North Jamieview, AP",2021-04-08,https://realpython.github.io/fake-jobs/jobs/pr...,Traditional page a although for study anyone. ...
